# 1D TFIM — HVA Experiment 

Self-contained notebook to construct the **Transverse-Field Ising Model (TFIM)** in 1D and the
**Hamiltonian Variational Ansatz (HVA)** circuits used by the pipeline, for **N = 10, 20, 30 qubits**.

This notebook is standalone (only depends on `qiskit` and `numpy`) but is numerically identical to the
production code in `qmbp_simulation.models.hamiltonian.HamiltonianBuilder.build` and
`qmbp_simulation.circuits.hva.HVACircuitBuilder.create`.

**Purpose:** hand off the exact Hamiltonian + circuit definitions so references can be prepared with
Exact Diagonalization (ED) and DMRG.

**Sweep parameters:** `h in {0.5, 1.0, 2.0}` (h=1.0 is the 1D critical point; 0.5 ordered, 2.0 paramagnetic),
`J = 1.0`, `p_layers = 1`, open boundary conditions.


## 1. The Hamiltonian (full description)

The 1D transverse-field Ising model on an **open chain** of $N$ qubits:

$$
H \;=\; -\,J \sum_{i=0}^{N-2} Z_i Z_{i+1} \;-\; h \sum_{i=0}^{N-1} X_i
$$

**Conventions (must be matched by ED / DMRG references):**

| Item | Value |
|------|-------|
| Coupling sign | **Ferromagnetic**: interaction term is $-J\,Z_iZ_{i+1}$ with $J=1$ |
| Field sign | Transverse field term is $-h\,X_i$ |
| Boundary conditions | **Open** (no $Z_{N-1}Z_0$ term) |
| Edges | $(i, i{+}1)$ for $i = 0 \dots N-2$ → exactly $N-1$ bonds |
| Qubit indexing | site $i$ ↔ qubit $i$; Qiskit little-endian (qubit 0 = rightmost tensor factor) |
| Critical point | $h_c = J = 1$ (1D TFIM, thermodynamic limit) |
| $\mathbb{Z}_2$ symmetry | $\prod_i X_i$ commutes with $H$ |

The operator is built with `SparsePauliOp.from_sparse_list`, identical to the production
`HamiltonianBuilder.build()`.


In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

print("qiskit:", __import__("qiskit").__version__)

# Experiment configuration
J = 1.0
H_VALUES = [0.5, 1.0, 2.0]     # h=1.0 primary (critical); 0.5 ordered, 2.0 paramagnetic
N_QUBITS = [10, 20, 30]
P_LAYERS = 1
PERIODIC = False               # open boundary conditions

## 2. Lattice — 1D chain edges

Open chain: bonds $(i, i{+}1)$. This mirrors `generate_chain_1d(n, periodic=False)`.


In [ ]:
def generate_chain_1d(n: int, periodic: bool = False) -> list[tuple[int, int]]:
    """Open (or periodic) 1D chain edge list. Matches qmbp_simulation."""
    edges = [(i, i + 1) for i in range(n - 1)]
    if periodic:
        edges.append((n - 1, 0))
    return edges


for n in N_QUBITS:
    e = generate_chain_1d(n, PERIODIC)
    print(f"N={n:2d}: {len(e)} bonds, first={e[:3]}, last={e[-1]}")

## 3. Build the TFIM Hamiltonian

$H = -J \sum Z_iZ_{i+1} - h \sum X_i$ as a `SparsePauliOp`, identical to `HamiltonianBuilder.build()`.


In [ ]:
def build_tfim_1d(n: int, h: float, J: float = 1.0, periodic: bool = False) -> SparsePauliOp:
    """Build H = -J sum_{(i,j)} Z_i Z_j - h sum_i X_i as a SparsePauliOp.

    Identical construction to qmbp_simulation.models.hamiltonian.HamiltonianBuilder.build.
    """
    edges = generate_chain_1d(n, periodic)
    terms: list[tuple[str, list[int], complex]] = []
    # ZZ interaction on each bond (ferromagnetic: -J)
    for (i, j) in edges:
        terms.append(("ZZ", [i, j], -J))
    # Transverse field on each site (-h X)
    for site in range(n):
        terms.append(("X", [site], -h))
    H = SparsePauliOp.from_sparse_list(terms, num_qubits=n)
    # Hermiticity sanity check
    assert np.allclose((H - H.adjoint()).simplify().coeffs, 0.0), "H not Hermitian"
    return H


# Demonstrate for N=10 at each h
for h in H_VALUES:
    H = build_tfim_1d(10, h, J)
    print(f"N=10, h={h}: {len(H)} Pauli terms (expect {10-1} ZZ + 10 X = 19)")

### 3.1 Explicit Pauli-term listing (unambiguous reference for ED/DMRG)

Printed term-by-term for a small case so the ED/DMRG builders can be validated against the exact
operator string, coefficient, and qubit indices.


In [ ]:
def print_pauli_terms(H: SparsePauliOp, max_terms: int = 40) -> None:
    labels = H.paulis.to_labels()
    coeffs = H.coeffs
    print(f"{len(H)} terms (qubit order: leftmost char = qubit {H.num_qubits-1} ... rightmost = qubit 0)")
    for k, (lab, c) in enumerate(zip(labels, coeffs)):
        if k >= max_terms:
            print(f"  ... ({len(H) - max_terms} more)")
            break
        print(f"  {c.real:+.3f}  {lab}")


print("=== 1D TFIM, N=6, h=1.0 (small illustration) ===")
print_pauli_terms(build_tfim_1d(6, 1.0, J))

## 4. The HVA circuit

Matching `HVACircuitBuilder.create()` exactly:

- **Initial state:** $|+\rangle^{\otimes N}$ (a Hadamard on every qubit).
- **Per layer $\ell$:** `RZZ(2 theta_zz)` on each chain bond, then `RX(2 theta_x)` on each qubit.
- **Parameter count:** $2p$ total; ordering is `[theta_zz^(0), theta_x^(0), theta_zz^(1), theta_x^(1), ...]`.
- **Gate convention:** the physical factor of 2, i.e. `rzz(2*theta)` = $e^{-i\,\theta\,Z_iZ_j}$ and `rx(2*theta)` = $e^{-i\,\theta X}$.

This is the ansatz whose parameters the GNN predicts.


In [ ]:
def build_hva_tfim_1d(n: int, p_layers: int, periodic: bool = False):
    """Build the HVA circuit for a 1D TFIM chain.

    Matches qmbp_simulation.circuits.hva.HVACircuitBuilder.create:
      - |+>^N initial state (Hadamard layer)
      - per layer: RZZ(2*theta_zz) on edges, then RX(2*theta_x) on all qubits
      - 2*p_layers parameters, ordered [theta_zz_0, theta_x_0, theta_zz_1, ...]

    Returns
    -------
    (qc, theta) : (QuantumCircuit, ParameterVector of length 2*p_layers)
    """
    edges = generate_chain_1d(n, periodic)
    if not edges:
        raise ValueError(f"No edges for N={n}")

    qc = QuantumCircuit(n)
    theta = ParameterVector("theta", 2 * p_layers)

    # Initial state |+>^N
    qc.h(range(n))

    for layer in range(p_layers):
        theta_zz = theta[layer * 2]
        theta_x = theta[layer * 2 + 1]
        # e^{-i theta_zz H_ZZ}
        for (i, j) in edges:
            qc.rzz(2 * theta_zz, i, j)
        # e^{-i theta_x H_X}
        for i in range(n):
            qc.rx(2 * theta_x, i)

    return qc, theta


qc10, th10 = build_hva_tfim_1d(10, P_LAYERS)
print(f"N=10, p={P_LAYERS}: {qc10.num_parameters} params, depth={qc10.depth()}, "
      f"gate counts={dict(qc10.count_ops())}")
qc10.draw("text", fold=120)

### 4.1 Circuits for N = 10, 20, 30

Built at `p_layers = 1`. N=30 statevector simulation is infeasible ($2^{30}$ amplitudes) — the circuit
and Hamiltonian are constructed for handoff to DMRG / MPS, but no in-notebook diagonalization is done there.


In [ ]:
circuits = {}
for n in N_QUBITS:
    qc, theta = build_hva_tfim_1d(n, P_LAYERS)
    circuits[n] = (qc, theta)
    ops = dict(qc.count_ops())
    print(f"N={n:2d}: params={qc.num_parameters}, depth={qc.depth()}, "
          f"n_rzz={ops.get('rzz', 0)}, n_rx={ops.get('rx', 0)}, n_h={ops.get('h', 0)}")

## 5. Exact-Diagonalization cross-check (N=10)

A ground-truth energy for the ED/DMRG teams to validate against. We build the dense matrix from the
`SparsePauliOp` and take the lowest eigenvalue with `numpy.linalg.eigvalsh`. Feasible up to ~N=12–14 in
a notebook; here we do N=10 for all three h-values.

**These numbers are the reference $E_0$ your ED/DMRG must reproduce (open BC, J=1).**


In [ ]:
def exact_ground_energy(n: int, h: float, J: float = 1.0, periodic: bool = False) -> float:
    H = build_tfim_1d(n, h, J, periodic)
    mat = H.to_matrix()  # dense 2^n x 2^n complex
    evals = np.linalg.eigvalsh(mat)
    return float(evals[0])


print("Exact ground-state energy E0 (1D TFIM, open BC, J=1.0)")
print(f"{'N':>3} {'h':>5} {'E0':>16} {'E0/N':>12}")
for n in [10]:
    for h in H_VALUES:
        e0 = exact_ground_energy(n, h, J)
        print(f"{n:>3} {h:>5.2f} {e0:>16.10f} {e0/n:>12.8f}")

## 6. Summary of what to reproduce in ED / DMRG

**Hamiltonian** (identical for all N; only the number of sites changes):

$$H = -\sum_{i=0}^{N-2} Z_i Z_{i+1} \;-\; h\sum_{i=0}^{N-1} X_i, \qquad J = 1,\ \ h \in \{0.5, 1.0, 2.0\}$$

- Open boundary conditions (no wrap-around bond).
- Ferromagnetic $-Z_iZ_{i+1}$ coupling, transverse field $-hX_i$.
- Pauli operators are the standard $2\times2$ matrices; $Z=\mathrm{diag}(1,-1)$, $X=\begin{psmallmatrix}0&1\\1&0\end{psmallmatrix}$.
- Qubit/site indexing is $0 \dots N-1$; Qiskit uses little-endian tensor ordering, but the energy
  spectrum is basis-order-independent, so a standard $Z=\mathrm{diag}(1,-1)$ ED build gives the same eigenvalues.

**HVA circuit** (the state whose energy is minimized):
- $|+\rangle^{\otimes N}$, then for each of $p$ layers: $\prod_{\text{bonds}} e^{-i\theta_{zz} Z_iZ_j}$ followed by
  $\prod_i e^{-i\theta_x X_i}$.
- $2p$ variational parameters.

**Reference values delivered:** exact $E_0$ for N=10 at $h\in\{0.5,1.0,2.0\}$ (Section 5). For N=20 use
sparse ED (`scipy.sparse.linalg.eigsh`, feasible to N≈22); for N=30 use DMRG/MPS ($\chi=64$ is validated
exact for 1D TFIM at any N in this project).


## 7. (Optional) Close the loop with the GNN

Everything above is **standalone** and does not touch the GNN: the HVA circuit is built with an
*unbound* `ParameterVector` (symbolic angles), and the only energy shown is the exact $E_0$ from
diagonalizing $H$. **No quantum state is materialized above** — not random, not predicted.

This section (disabled by default via `USE_GNN = False`) closes the loop:

1. Load the best zoo model for `chain_1d`, `p=1` — `load_best_model_for(...)`.
2. Predict the HVA angles $\theta$ for a given $h$ from the graph.
3. Bind $\theta$ into the circuit — `qc.assign_parameters(...)`.
4. Evaluate $\langle H\rangle$ of the resulting state and compare to $E_0$ (report $|\Delta E|$ and $\Delta E/\text{gap}$).

**Requires** `qmbp_simulation` + `torch` + a trained checkpoint, so enabling it breaks the standalone
property. Leave `USE_GNN = False` to keep the notebook runnable anywhere.

> **Important — model regime.** The chain_1d zoo models are trained mostly in the **paramagnetic
> regime** ($h \gtrsim 2.5$). The sanity-check values $h \in \{0.5, 1.0\}$ sit at/below the critical
> point $h_c = 1$, **outside** the model's training range, so predictions there are expected to be
> poor. This is a property of the trained model, not of the loop below — the cell prints the honest
> $|\Delta E|$ so you can see exactly where the model works and where it does not. The production
> ansatz here is **bond-resolved** (`create_bond_resolved`, $(n_\text{edges}+N)\cdot p$ parameters),
> which is what the zoo predicts — not the 2-parameter global HVA from Section 4.


In [ ]:
USE_GNN = False  # set True to load the GNN and evaluate <H> of the predicted state (needs torch + qmbp_simulation)
GNN_N = 10        # system size for the demo (N=10 has an exact E0 above)

In [ ]:
if USE_GNN:
    import numpy as np
    import torch
    from qiskit.quantum_info import Statevector

    from qmbp_simulation.circuits.hva import HVACircuitBuilder
    from qmbp_simulation.models.hamiltonian import HamiltonianBuilder, make_lattice
    from qmbp_simulation.predictors.model_zoo import load_best_model_for
    from qmbp_simulation.predictors.unified_graph import build_unified_bond_resolved_graph

    N = GNN_N
    model, entry, source = load_best_model_for(
        "chain_1d", model="tfim_bond_resolved", p_layers=P_LAYERS, n_target=N
    )
    print(f"model: {entry.checkpoint_file}")
    print(f"source={source}  pass_rate={entry.pass_rate:.0%}  trained h_range={entry.h_range}")
    print(f"(sanity-check h in {H_VALUES}; anything below the trained h_range is extrapolation)")

    hb = HamiltonianBuilder()
    builder = HVACircuitBuilder()
    model.eval()

    print(f"{'h':>5} {'E_GNN':>12} {'E0':>12} {'|dE|':>10} {'dE/gap':>10}")
    for h in H_VALUES:
        lat = make_lattice("chain_1d", N, J=J, h=h, periodic=PERIODIC)
        # 1-2. Predict theta from the unified Hamiltonian+circuit graph
        graph = build_unified_bond_resolved_graph(
            lat, h_value=h, p_layers=P_LAYERS, include_circuit_nodes=True
        )
        with torch.no_grad():
            theta = model(graph).numpy().flatten()
        theta = np.clip(theta, -np.pi, np.pi)  # valid HVA range

        # 3. Bind theta into the bond-resolved HVA circuit (what the zoo predicts)
        qc, tv = builder.create_bond_resolved(N, P_LAYERS, lat)
        assert theta.shape[0] == qc.num_parameters, (theta.shape, qc.num_parameters)
        bound = qc.assign_parameters({tv[i]: float(theta[i]) for i in range(len(theta))})

        # 4. Evaluate <H> and compare to exact E0
        H = hb.build(lat)
        E_gnn = float(Statevector(bound).expectation_value(H).real)
        ev = np.linalg.eigvalsh(H.to_matrix())
        E0, gap = float(ev[0]), float(ev[1] - ev[0])
        print(f"{h:>5.2f} {E_gnn:>12.5f} {E0:>12.5f} {abs(E_gnn-E0):>10.5f} {abs(E_gnn-E0)/gap:>10.4f}")
else:
    print("USE_GNN is False - skipping GNN inference (notebook stays standalone).")
    print("Set USE_GNN = True above to load the zoo model and evaluate the predicted state.")